# Part 2 – Deep Learning models for network anomaly detection (CICIDS2018 and NSL-KDD)

Implements the second part of the report's methodology:

* **Datasets** – **CICIDS2018** (sequential, explicit `Timestamp` attribute, wide range of attack classes) and **NSL-KDD**.
* **Preprocessing** – the datasets' flaws (sparsity, severe class imbalance, high dimensionality) are handled with
  **categorical data encoding (embedding layers)**, a **focal loss** function and an **autoencoder**.
* **CNN** – 1D sequence as input to the convolution layers; multiple **Conv1D** layers with **ReLU**, followed by **max pooling**;
  kernel size **4**, **128** filters; **Adam**; learning rate **0.001** (CICIDS2018) / **0.003** (NSL-KDD);
  dropout **0.4** (CICIDS2018) / **0.3** (NSL-KDD).
* **LSTM** – sequence length 50–100 timesteps (CICIDS2018) and 30–50 (NSL-KDD); 128–256 / 64–128 units per layer;
  3–4 / 2–3 stacked layers. This notebook uses the lower end of each range: **50 steps, 128 units, 3 layers** (CICIDS2018) and
  **30 steps, 64 units, 2 layers** (NSL-KDD).
* **RNN** – reported in Tables 3 and 4; built like the LSTM with SimpleRNN layers.
* Task: binary detection (0 = benign/normal, 1 = attack), as in Fig 3 / Fig 4.

Outputs: `results/part2_*.csv`, `figures/part2_*.png`.

In [ ]:
import os, glob, time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
EPOCHS = 10          # with early stopping on a validation split
BATCH_SIZE = 256
EMB_DIM = 4          # embedding size per categorical feature
os.makedirs("results", exist_ok=True); os.makedirs("figures", exist_ok=True)

## Shared building blocks – autoencoder, embeddings, focal loss, CNN / LSTM / RNN, sequence batches

In [ ]:
def train_autoencoder(X_fit, latent_dim, name):
    # Dense autoencoder used for dimensionality reduction of the numeric features.
    d = X_fit.shape[1]
    inp = keras.Input((d,))
    z = layers.Dense(64, activation="relu")(inp)
    z = layers.Dense(latent_dim, activation="relu", name="latent")(z)
    out = layers.Dense(64, activation="relu")(z)
    out = layers.Dense(d, activation="linear")(out)
    ae = keras.Model(inp, out); ae.compile(optimizer="adam", loss="mse")
    ae.fit(X_fit, X_fit, epochs=5, batch_size=1024, validation_split=0.1, verbose=2)
    return keras.Model(inp, z, name=f"{name}_encoder")

def encode(encoder, X, chunk=500_000):
    return np.concatenate([encoder.predict(X[i:i + chunk], batch_size=8192, verbose=0) for i in range(0, len(X), chunk)]).astype(np.float32)

def embed(cat_in, vocab_sizes, seq):
    # One embedding layer per categorical feature; output is concatenated with the numeric features.
    embs = []
    for j, v in enumerate(vocab_sizes):
        col = layers.Lambda(lambda t, j=j: t[..., j])(cat_in)
        embs.append(layers.Embedding(v, EMB_DIM)(col))
    return layers.Concatenate(axis=-1)(embs) if len(embs) > 1 else embs[0]

def focal_loss():
    return keras.losses.BinaryFocalCrossentropy(gamma=2.0)

def build_cnn(num_dim, vocab_sizes, lr, dropout):
    num_in = keras.Input((num_dim,), name="num"); cat_in = keras.Input((len(vocab_sizes),), dtype="int32", name="cat")
    x = layers.Concatenate()([num_in, embed(cat_in, vocab_sizes, seq=False)])
    x = layers.Reshape((-1, 1))(x)                                          # 1D sequence as convolution input
    x = layers.Conv1D(128, 4, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 4, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model([num_in, cat_in], out, name="CNN")
    m.compile(optimizer=keras.optimizers.Adam(lr), loss=focal_loss(), metrics=["accuracy"])
    return m

def build_recurrent(kind, seq_len, num_dim, vocab_sizes, units, n_layers, dropout):
    Cell = layers.LSTM if kind == "LSTM" else layers.SimpleRNN
    num_in = keras.Input((seq_len, num_dim), name="num"); cat_in = keras.Input((seq_len, len(vocab_sizes)), dtype="int32", name="cat")
    x = layers.Concatenate()([num_in, embed(cat_in, vocab_sizes, seq=True)])
    for i in range(n_layers):
        x = Cell(units, return_sequences=i < n_layers - 1)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model([num_in, cat_in], out, name=kind)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss=focal_loss(), metrics=["accuracy"])
    return m

class Windows(keras.utils.PyDataset):
    # Sequence windows ending at the given row indices: rows [end-seq_len+1 .. end]; label = label of the last row.
    def __init__(self, num, cat, y, ends, seq_len, batch_size, shuffle):
        super().__init__()
        self.num, self.cat, self.y, self.ends, self.offs = num, cat, y, ends, np.arange(-seq_len + 1, 1)
        self.bs, self.shuffle = batch_size, shuffle
        self.order = np.arange(len(ends)); self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.ends) / self.bs))
    def __getitem__(self, i):
        e = self.ends[self.order[i * self.bs:(i + 1) * self.bs]]
        w = e[:, None] + self.offs
        return {"num": self.num[w], "cat": self.cat[w]}, self.y[e]
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.order)

def valid_window_ends(group_ids, seq_len):
    # Window end indices whose whole window lies inside one group (one file / one day).
    ok = np.zeros(len(group_ids), dtype=bool)
    ok[seq_len - 1:] = group_ids[seq_len - 1:] == group_ids[:len(group_ids) - seq_len + 1]
    return np.flatnonzero(ok)

def evaluate(name, y_true, prob, dataset):
    y_pred = (prob.ravel() >= 0.5).astype(int)
    print(f"{name} – {dataset}")
    print(classification_report(y_true, y_pred, digits=2))
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
    return {"Model": name, "Precision": p, "Recall": r, "F1-score": f, "Accuracy": accuracy_score(y_true, y_pred)}

def plot_history(histories, dataset, fname):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for name, h in histories.items():
        axes[0].plot(h["accuracy"], "-o", label=f"{name} train"); axes[0].plot(h["val_accuracy"], "--o", label=f"{name} val")
        axes[1].plot(h["loss"], "-o", label=f"{name} train"); axes[1].plot(h["val_loss"], "--o", label=f"{name} val")
    for ax, t in zip(axes, ["Accuracy", "Loss"]):
        ax.set_title(f"{dataset} – {t} over epochs"); ax.set_xlabel("Epoch"); ax.grid(True); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(fname, dpi=120); plt.show()

def table(rows, title, csv):
    t = pd.DataFrame(rows); t.to_csv(csv, index=False)
    shown = t.copy()
    for c in ["Precision", "Recall", "F1-score", "Accuracy"]:
        shown[c] = (shown[c] * 100).map("{:.1f}%".format)
    print(title); display(shown)

early = lambda: [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]

---
# A. NSL-KDD

Standard split: `KDDTrain+` for training, `KDDTest+` for testing. Records are used in file order to form sequences.
Categorical features (`protocol_type`, `service`, `flag`) are encoded with embedding layers.

In [ ]:
KDD_COLS = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
            "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
            "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count", "srv_count",
            "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
            "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
            "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
            "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"]
CAT = ["protocol_type", "service", "flag"]
read = lambda f: pd.read_csv(f, sep=None, engine="python", header=None, names=KDD_COLS)
kdd_tr, kdd_te = read("datasets/nsl-kdd/KDDTrain+.txt"), read("datasets/nsl-kdd/KDDTest+.txt")
print(kdd_tr.shape, kdd_te.shape)
print(kdd_tr["label"].eq("normal").map({True: "normal", False: "attack"}).value_counts())

NUM = [c for c in KDD_COLS if c not in CAT + ["label", "difficulty"]]
ytr_k = (kdd_tr["label"] != "normal").astype(np.float32).values
yte_k = (kdd_te["label"] != "normal").astype(np.float32).values

# categorical encoding (index 0 = category not seen in training)
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(kdd_tr[c].unique()))} for c in CAT}
vocab_k = [len(vocab[c]) + 1 for c in CAT]
cat_tr_k = np.stack([kdd_tr[c].map(vocab[c]).fillna(0).astype(np.int32) for c in CAT], 1)
cat_te_k = np.stack([kdd_te[c].map(vocab[c]).fillna(0).astype(np.int32) for c in CAT], 1)

# numeric features: log-scaling of the sparse heavy-tailed counts, then standardisation
Xtr = np.log1p(kdd_tr[NUM].clip(lower=0).values.astype(np.float64)); Xte = np.log1p(kdd_te[NUM].clip(lower=0).values.astype(np.float64))
mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd == 0] = 1
Xtr = ((Xtr - mu) / sd).astype(np.float32); Xte = ((Xte - mu) / sd).astype(np.float32)

### Autoencoder – dimensionality reduction of the 38 numeric NSL-KDD features to 16

In [ ]:
enc_k = train_autoencoder(Xtr, 16, "nslkdd")
num_tr_k, num_te_k = encode(enc_k, Xtr), encode(enc_k, Xte)
print(num_tr_k.shape, num_te_k.shape)

### CNN – NSL-KDD (lr 0.003, dropout 0.3)

In [ ]:
hist_k, rows_k = {}, []
cnn_k = build_cnn(num_tr_k.shape[1], vocab_k, lr=0.003, dropout=0.3); cnn_k.summary()
t0 = time.time()
h = cnn_k.fit({"num": num_tr_k, "cat": cat_tr_k}, ytr_k, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=early(), verbose=2)
print(f"training time {time.time() - t0:.0f}s"); hist_k["CNN"] = h.history
rows_k.append(evaluate("CNN", yte_k, cnn_k.predict({"num": num_te_k, "cat": cat_te_k}, batch_size=4096, verbose=0), "NSL-KDD"))

### LSTM and RNN – NSL-KDD (sequence length 30, 64 units, 2 stacked layers)

In [ ]:
SEQ_K, UNITS_K, LAYERS_K, DROP_K = 30, 64, 2, 0.3
ends_tr = valid_window_ends(np.zeros(len(num_tr_k), int), SEQ_K)
ends_te = valid_window_ends(np.zeros(len(num_te_k), int), SEQ_K)
e_fit, e_val = train_test_split(ends_tr, test_size=0.1, random_state=SEED)
for kind in ["LSTM", "RNN"]:
    m = build_recurrent(kind, SEQ_K, num_tr_k.shape[1], vocab_k, UNITS_K, LAYERS_K, DROP_K); m.summary()
    t0 = time.time()
    h = m.fit(Windows(num_tr_k, cat_tr_k, ytr_k, e_fit, SEQ_K, BATCH_SIZE, True), epochs=EPOCHS,
              validation_data=Windows(num_tr_k, cat_tr_k, ytr_k, e_val, SEQ_K, 2048, False), callbacks=early(), verbose=2)
    print(f"training time {time.time() - t0:.0f}s"); hist_k[kind] = h.history
    prob = m.predict(Windows(num_te_k, cat_te_k, yte_k, ends_te, SEQ_K, 2048, False), verbose=0)
    rows_k.append(evaluate(kind, yte_k[ends_te], prob, "NSL-KDD"))

### Results – Fig 3 / Table 3 (NSL-KDD)

In [ ]:
table(rows_k, "Table 3 - Performance metrics of models on NSL-KDD", "results/part2_nslkdd_results.csv")
plot_history(hist_k, "NSL-KDD", "figures/part2_nslkdd_training.png")

---
# B. CICIDS2018

The *Processed Traffic Data for ML Algorithms* CSVs of nine capture days (Tuesday 20-02-2018 not included).
Each day is sorted by its `Timestamp` so consecutive flows form the sequences. `Protocol` is encoded with an embedding layer.

In [ ]:
files18 = sorted(glob.glob("datasets/cicids2018/*_TrafficForML_CICFlowMeter.csv"))
nums, cats, ys, days = [], [], [], []
for d, f in enumerate(files18):
    df = pd.read_csv(f, low_memory=False)
    df = df[df["Label"] != "Label"]                                   # repeated header rows inside some files
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
    feats = [c for c in df.columns if c not in ("Timestamp", "Label", "Protocol")]
    X = df[feats].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1) & df["Timestamp"].notna()
    order = df.loc[ok, "Timestamp"].values.argsort(kind="stable")
    nums.append(X[ok].values.astype(np.float32)[order])
    cats.append(pd.to_numeric(df.loc[ok, "Protocol"]).values.astype(np.int32)[order])
    ys.append((df.loc[ok, "Label"] != "Benign").values.astype(np.float32)[order])
    days.append(np.full(ok.sum(), d, dtype=np.int16))
    print(os.path.basename(f), int(ok.sum()), "flows,", f"{ys[-1].mean():.1%} attack")
    del df, X
X18 = np.concatenate(nums); y18 = np.concatenate(ys); day18 = np.concatenate(days)
proto_vals = sorted(set(np.concatenate(cats)))
cat18 = np.searchsorted(proto_vals, np.concatenate(cats)).astype(np.int32)[:, None] + 1
vocab_18 = [len(proto_vals) + 1]
del nums, cats, ys, days
print("total:", X18.shape, f"{y18.mean():.1%} attack")

In [ ]:
# log-scaling of the heavy-tailed, sparse flow statistics, then standardisation (done in place, chunk by chunk)
for i in range(0, len(X18), 1_000_000):
    blk = X18[i:i + 1_000_000]
    np.log1p(np.clip(blk, 0, None), out=blk)
mu18 = X18.mean(0, dtype=np.float64); sd18 = X18.std(0, dtype=np.float64); sd18[sd18 == 0] = 1
for i in range(0, len(X18), 1_000_000):
    X18[i:i + 1_000_000] = ((X18[i:i + 1_000_000] - mu18) / sd18).astype(np.float32)

### Train / test windows

Windows of 50 consecutive flows lie inside a single day. 300,000 windows are sampled for training and 75,000 for testing
(CPU training budget); the CNN uses the last flow of each of the same windows.

In [ ]:
SEQ_18, UNITS_18, LAYERS_18, DROP_18 = 50, 128, 3, 0.4
ends18 = valid_window_ends(day18, SEQ_18)
rng = np.random.default_rng(SEED)
pick = rng.choice(ends18, size=375_000, replace=False)
e_tr, e_te = train_test_split(pick, test_size=75_000, random_state=SEED, stratify=y18[pick])
e_fit, e_val = train_test_split(e_tr, test_size=0.1, random_state=SEED)
print("train windows:", len(e_fit), "val:", len(e_val), "test:", len(e_te), f"attack share (test): {y18[e_te].mean():.1%}")

### Autoencoder – dimensionality reduction of the 77 numeric CICIDS2018 features to 32

In [ ]:
enc_18 = train_autoencoder(X18[rng.choice(e_tr, size=300_000, replace=False)], 32, "cicids2018")
num18 = encode(enc_18, X18)
del X18
print(num18.shape)

### CNN – CICIDS2018 (lr 0.001, dropout 0.4)

In [ ]:
hist_18, rows_18 = {}, []
cnn_18 = build_cnn(num18.shape[1], vocab_18, lr=0.001, dropout=0.4); cnn_18.summary()
t0 = time.time()
h = cnn_18.fit({"num": num18[e_fit], "cat": cat18[e_fit]}, y18[e_fit], validation_data=({"num": num18[e_val], "cat": cat18[e_val]}, y18[e_val]),
               epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=early(), verbose=2)
print(f"training time {time.time() - t0:.0f}s"); hist_18["CNN"] = h.history
rows_18.append(evaluate("CNN", y18[e_te], cnn_18.predict({"num": num18[e_te], "cat": cat18[e_te]}, batch_size=4096, verbose=0), "CICIDS2018"))

### LSTM and RNN – CICIDS2018 (sequence length 50, 128 units, 3 stacked layers)

In [ ]:
for kind in ["LSTM", "RNN"]:
    m = build_recurrent(kind, SEQ_18, num18.shape[1], vocab_18, UNITS_18, LAYERS_18, DROP_18); m.summary()
    t0 = time.time()
    h = m.fit(Windows(num18, cat18, y18, e_fit, SEQ_18, BATCH_SIZE, True), epochs=EPOCHS,
              validation_data=Windows(num18, cat18, y18, e_val, SEQ_18, 2048, False), callbacks=early(), verbose=2)
    print(f"training time {time.time() - t0:.0f}s"); hist_18[kind] = h.history
    prob = m.predict(Windows(num18, cat18, y18, e_te, SEQ_18, 2048, False), verbose=0)
    rows_18.append(evaluate(kind, y18[e_te], prob, "CICIDS2018"))

### Results – Fig 4 / Table 4 (CICIDS2018)

In [ ]:
table(rows_18, "Table 4 - Performance metrics of models on CICIDS-2018", "results/part2_cicids2018_results.csv")
plot_history(hist_18, "CICIDS2018", "figures/part2_cicids2018_training.png")